# Generating data for AI training

## Generate a specset

In [2]:
from pathlib import Path

# change path to the data here
p = Path("/d0/tp/scratch/specset_data/hela_ai")

from specset.utils import maxquant_to_specset

print("Create specset from MaxQuant search result...")

import shutil
if (p / 'txt.specset').exists:
    print(f"Output directory exists. Removing {p / 'txt.specset'}...")
    shutil.rmtree(p / 'txt.specset')

maxquant_to_specset(p / 'txt', p / 'txt.specset', additional_columns = {"Score" : "score", "Proteins" : "proteins", "Gene Names" : "genes"})


Create specset from MaxQuant search result...
Output directory exists. Removing /d0/tp/scratch/specset_data/hela_ai/txt.specset...
Reading msms.txt...
Reading evidence.txt...
Creating /d0/tp/scratch/specset_data/hela_ai/txt.specset...
n = 278280
Done.


## To AI data

Here we experiment with alphapeptdeep ecosystem. Install alphapeptdeep and alphabase
```
pip install "alphabase[full-stable]"
pip install "peptdeep[stable,gui-stable]"
```

Download and unzip the pretrained models. Adapt the folder path appropriately.

https://github.com/MannLabs/alphapeptdeep/releases/download/pre-trained-models/pretrained_models_v3.zip


In [2]:
import os
os.environ["TQDM_DISABLE"] = "1"                  # Suppress tqdm progress bar

import tqdm

import warnings
warnings.filterwarnings('ignore')

from specset.utils import specset_to_alphapeptdeep_string

from pathlib import Path
# change path to the data here
p = Path("/d0/tp/scratch/specset_data/hela_ai")

print("Create alphapeptdeep stream...")
str_stream = specset_to_alphapeptdeep_string(p / 'txt.specset')

dfs = []
frag_inten_dfs = []
from alphabase.spectral_library.reader import LibraryReaderBase
_lib = LibraryReaderBase()

n = 0
for psm in str_stream:    
    n += 1
    if n % 1000 == 0:
        print(f"n = {n}", end = '\r')
    a = _lib.import_file(psm)
    dfs.append(a)
    frag_inten_dfs.append(_lib.fragment_intensity_df)

print(f"Number of PSMs = {n}")

import pickle
f = open(p / 'txt.specset.dfs.pckl', 'wb')
pickle.dump(dfs, f)
f.close()

f = open(p / 'txt.specset.frag_inten_dfs.pckl', 'wb')
pickle.dump(frag_inten_dfs, f)
f.close()

print("Done.")

Create alphapeptdeep stream...


/d0/tp/aiproteomics/venv_specset/lib/python3.10/site-packages/alphabase/psm_reader/psm_reader.py:168: FutureWarning: Passed unknown arguments to LibraryReaderBase (fixed_C57=False) will be forbidden in alphabase>1.5.0.
  warnings.warn(


Number of PSMs = 278280
Done.


## Refine the generic MS2 model

In [ ]:
from pathlib import Path

# change path to the data here
p = Path("/d0/tp/scratch/specset_data/hela_ai")

import torch # noqa: 401, to prevent crash in Mac Arm
import os
import warnings

import pickle

print('Load PSMs...')
with open(p / 'txt.specset.dfs.pckl', 'rb') as fp:
    dfs = pickle.load(fp)
with open(p / 'txt.specset.frag_inten_dfs.pckl', 'rb') as fp:
    frag_inten_dfs = pickle.load(fp)

print('Prepare alphapeptdeep data...')
from alphabase.peptide.fragment import concat_precursor_fragment_dataframes

psm_df, frag_df = concat_precursor_fragment_dataframes(dfs, frag_inten_dfs)

# fine tune
from peptdeep.model.ms2 import pDeepModel

# change path to the alphapeptdeep data here
model_path = p / 'alphapeptdeep_data/generic/ms2.pth'

model_interface = pDeepModel(override_from_weights=True)
model_interface.load(model_path)

supported_fragment_types = model_interface.model.supported_charged_frag_types
print(f"Fragment types supported by the model: {supported_fragment_types}")

psm_df['nce'] = 30.0
psm_df['instrument'] = 'timsTOF'

print('\nFine-tune the generic MS2 model...\n')

model_interface.train(precursor_df=psm_df, fragment_intensity_df=frag_df.loc[:, supported_fragment_types], 
                      lr= 0.0001,
                      batch_size=512,
                      warmup_epoch=10, epoch=20, verbose=1)

#save model
model_interface.save(str(p / 'transfer_ms2_hela/transfer_ms2.pth'))
print(f"Fine-tuned model saved in: {p / 'transfer_ms2_hela/transfer_ms2.pth'}\n")

#####
print('\nGenerate library using adapted MS2 model...\n')

from peptdeep.pipeline_api import global_settings, generate_library

fasta = str(p / 'uniprot_Human.fasta')

lib_settings = global_settings['library']

lib_settings['infile_type'] = 'fasta'
lib_settings['infiles'] = [fasta]
lib_settings['fasta']['add_contaminants'] = False
lib_settings['output_tsv']['enabled'] = True
lib_settings['output_folder'] = str(p / 'lib_transfer_ms2_hela')


mgr_settings = global_settings['model_mgr']

mgr_settings['external_ms2_model'] = str(p / 'transfer_ms2_hela/transfer_ms2.pth')
mgr_settings['external_rt_model'] = str(p / 'alphapeptdeep_data/generic/rt.pth')
mgr_settings['external_ccs_model'] = str(p / 'alphapeptdeep_data/generic/ccs.pth')

generate_library()
print(f"Output library saved in: {p / 'lib_transfer_ms2_hela'}\n")

#####
print('\nGenerate library using the default model...\n')

lib_settings['output_folder'] = str(p / 'lib_default_hela')

mgr_settings['external_ms2_model'] = str(p / 'alphapeptdeep_data/generic/ms2.pth')
mgr_settings['external_rt_model'] = str(p / 'alphapeptdeep_data/generic/rt.pth')
mgr_settings['external_ccs_model'] = str(p / 'alphapeptdeep_data/generic/ccs.pth')

generate_library()
print(f"Output library saved in: {p / 'lib_default_hela'}\n")


Load PSMs...
Prepare alphapeptdeep data...
Fragment types supported by the model: ['b_z1', 'b_z2', 'y_z1', 'y_z2', 'b_modloss_z1', 'b_modloss_z2', 'y_modloss_z1', 'y_modloss_z2']
MS2 model adaptation...
2026-10-01 22:45:49> Training with fixed sequence length: 0
[Training] Epoch=1, lr=2e-05, loss=0.012417424656450748
[Training] Epoch=2, lr=3e-05, loss=0.011677501375966688
[Training] Epoch=3, lr=4e-05, loss=0.01157209052001698
[Training] Epoch=4, lr=5e-05, loss=0.011507885113946702
[Training] Epoch=5, lr=6e-05, loss=0.01143776534059853
[Training] Epoch=6, lr=7e-05, loss=0.011436086688027309
[Training] Epoch=7, lr=8e-05, loss=0.011463664902635572
[Training] Epoch=8, lr=9e-05, loss=0.011345848499127386
[Training] Epoch=9, lr=0.0001, loss=0.01134206377574357
[Training] Epoch=10, lr=0.0001, loss=0.01131974042458623
[Training] Epoch=11, lr=9.755282581475769e-05, loss=0.011409110729500913
[Training] Epoch=12, lr=9.045084971874738e-05, loss=0.01128862715334419
[Training] Epoch=13, lr=7.9389262